<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Introduction to Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">1. Apache Spark Runtime Architecture</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 1.3 Demo - Exploring Spark Architecture in Databricks
### Duration - 11 Minutes

In this demonstration, we explore how Spark's architecture components manifest in a Databricks environment and how to monitor them using the various UIs available to us.

## Learning Objectives

**By the end of this demo, you will be able to:**

- Identify key Spark architecture components in a Databricks cluster
- Navigate the Spark UI to monitor application execution
- Describe how Databricks implements Spark's cluster management

## REQUIRED - SELECT CLASSIC COMPUTE

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select your <strong>Classic Compute</strong> cluster from the compute drop-down at the top-right of the notebook. Be aware that <strong>Serverless</strong> is selected by default.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li>If your cluster is not shown in the drop-down, select <strong>More</strong>, then choose your cluster in the <strong>Attach to an existing compute resource</strong> dialog.</li>
<li>If your cluster has terminated, open <strong>Compute</strong> in the left navigation, start your cluster, then return here and attach it.</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This course was developed and tested on <strong>Databricks Runtime 17.3 LTS (Apache Spark 4.0)</strong>. Several demonstrations explore the Spark UI and cluster internals, which are not exposed on Serverless compute.</p>
</div>
</div>
</div>

## A. Introduction to Notebooks

We will run all of the demos and exercises for this course in notebooks. Notebooks are interactive documents that combine live code, visualizations, narrative text, and outputs in a single document.

Key features of Databricks notebooks:

- Code execution: run code in multiple languages (Python, SQL, R, Scala)
- Rich text formatting: support for Markdown to create well-documented analyses
- Cell-based structure: code and content are organized into executable cells
- Interactive visualizations: direct plotting and charting of results
- Collaboration: share and work together on notebooks in real time

The default language for a notebook is shown in the toolbar above. The notebooks in this course default to SQL, and we use magic commands to run other languages within the same notebook:

- Use `%python` to execute Python code
- Use `%sql` to execute SQL queries
- Use `%r` to execute R code
- Use `%scala` to execute Scala code

The `%run` magic command is particularly useful - it executes another notebook within your current notebook, enabling modular code organization and reuse. The demos in this course use `%run` to invoke their classroom setup notebooks.

## B. The SparkSession and SparkContext

<p style="font-size: 1em; line-height: 1.6; color: #333">The <code>SparkSession</code> is automatically instantiated as <code>spark</code> in Databricks notebooks connected to a cluster. The legacy <code>SparkContext</code> sits beneath it. On dedicated-access classic clusters you can reach it as <code>spark.sparkContext</code> or <code>sc</code>, but standard-access (shared) and Serverless compute use Spark Connect, where direct access to the <code>SparkContext</code> and the RDD API is not available.</p>

In [0]:
%python
# Display the SparkSession
spark

In [0]:
%python
# Confirm the Spark Version
print(f"This cluster is running Apache Spark {spark.version}")

<div style="font-size: 1em; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#8505;&#65039;</span>
<div>
<strong style="color: #0d47a1; font-size: 1.1em">Spark 4.0</strong>
<p style="margin: 8px 0 0 0; color: #333">The version printed above should start with <code>4.</code> - Databricks Runtime 17.x is built on Apache Spark 4.0. Everything in this course uses Spark 4 behavior, including ANSI SQL mode by default.</p>
</div>
</div>
</div>

In [0]:
%python
# Get the Application ID
# spark.conf is Spark Connect safe; spark.sparkContext.applicationId is not
application_id = spark.conf.get("spark.app.id")
print(f"Application ID: {application_id}")

In [0]:
%python
# The SparkSession is a Managed Singleton
from pyspark.sql import SparkSession

# In Databricks the session already exists; getOrCreate() returns that same
# session rather than constructing a new one
print(f"getOrCreate() returns the existing session: {SparkSession.builder.getOrCreate() is spark}")
print(f"An active session is set: {SparkSession.getActiveSession() is not None}")

In [0]:
%python
# The Legacy SparkContext is Not Exposed on Spark Connect
# Standard-access (shared) and Serverless compute block direct JVM access
try:
    print(sc)
except Exception as e:
    print("SparkContext (sc) is not available on this compute.")
    print(f"{type(e).__name__}: {e}")

<div style="font-size: 1em; border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#128221;</span>
<div>
<strong style="color: #4a148c; font-size: 1.1em">The SparkContext is Legacy</strong>
<p style="margin: 8px 0 0 0; color: #333">The <code>SparkContext</code> (<code>sc</code>) predates the <code>SparkSession</code> and remains available only on dedicated-access classic clusters. On compute that uses Spark Connect - standard-access (shared) clusters and Serverless - the <code>SparkContext</code> and the RDD API are not available, which is why the cell above prints a message instead of failing. Modern Spark code works through the <code>SparkSession</code> and DataFrames.</p>
</div>
</div>
</div>

## C. Understanding Your Databricks Cluster

1. Right-click on the **Compute** menu item in the left navigation pane and select **Open link in new tab**
2. Locate your cluster in the **All-purpose compute** tab. Here you can see high-level information about the cluster including memory and cores
3. Click on the link to the cluster under the **Name** column. Here you can see more detailed information and access logs and metrics

## D. Creating and Monitoring a Spark Job

Let's create a simple job that will help us visualize the execution flow. Run the cell below to create a Spark job.

<div style="font-size: 1em; border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#128221;</span>
<div>
<strong style="color: #4a148c; font-size: 1.1em">Focus on the Execution, Not the Code</strong>
<p style="margin: 8px 0 0 0; color: #333">Don't focus too much on the code here. We want to explore the logs and metrics this job produces. The DataFrame operations themselves are covered in the next section.</p>
</div>
</div>
</div>

In [0]:
%python
# Run a Job with a Shuffle and a Cache
from pyspark.sql.functions import col, sum

# Create a large DataFrame to see parallelization in action
df = spark.range(0, 1000000)
df = df.withColumn("square", col("id") * col("id"))

# Force multiple stages with a shuffle operation
result = df.groupBy(col("id") % 100).agg(sum("square").alias("sum_squares"))

# Cache the result to see storage in the UI
result.cache()

# Force the computation with the collect() action
result.collect()

In [0]:
%python
# Trigger a Second Job
print(f"Number of groups: {result.count()}")

### Exploring the Spark UI

<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#128161;</span>
<div>
<strong style="color: #00695c; font-size: 1.1em">Open the Spark UI</strong>
<p style="margin: 8px 0 0 0; color: #333">Expand the <strong>Spark Jobs</strong> output under the cell above, then right-click the last <strong>View</strong> link and open it in a new tab.</p>
</div>
</div>
</div>

<p style="font-size: 1em; line-height: 1.6; color: #333">Now that we have an active job, click through the tabs below to explore the key areas of the Spark UI:</p>

<div style="margin: 16px 0">
<div id="sparkui-panel-jobs" style="display: block; opacity: 1; transition: opacity 0.3s ease; min-height: 130px; padding: 18px 22px; background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; margin-bottom: 12px">
<div style="font-size: 1.15em; font-weight: bold; color: #1B5162; margin: 0 0 8px 0">Jobs Tab</div>
<ul style="font-size: 1.05em; line-height: 1.5; color: #333; margin: 0; padding-left: 22px">
<li style="margin: 5px 0">Shows the DAG for our <code>groupBy</code> operation</li>
<li style="margin: 5px 0">Multiple stages appear due to the shuffle operation</li>
<li style="margin: 5px 0">Click the link inside the description of a stage to see task-level details</li>
</ul>
</div>
<div id="sparkui-panel-executors" style="display: block; opacity: 1; transition: opacity 0.3s ease; min-height: 130px; padding: 18px 22px; background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; margin-bottom: 12px">
<div style="font-size: 1.15em; font-weight: bold; color: #1B5162; margin: 0 0 8px 0">Executors Tab</div>
<ul style="font-size: 1.05em; line-height: 1.5; color: #333; margin: 0; padding-left: 22px">
<li style="margin: 5px 0">Lists all executors and their resource usage</li>
<li style="margin: 5px 0">Shows how many cores and how much memory each executor has</li>
<li style="margin: 5px 0">Demonstrates the Worker node distribution</li>
</ul>
</div>
<div id="sparkui-panel-storage" style="display: block; opacity: 1; transition: opacity 0.3s ease; min-height: 130px; padding: 18px 22px; background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; margin-bottom: 12px">
<div style="font-size: 1.15em; font-weight: bold; color: #1B5162; margin: 0 0 8px 0">Storage Tab</div>
<ul style="font-size: 1.05em; line-height: 1.5; color: #333; margin: 0; padding-left: 22px">
<li style="margin: 5px 0">Shows cached DataFrames</li>
<li style="margin: 5px 0">Displays memory usage across executors</li>
</ul>
</div>
</div>


<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">The architecture from the lecture (Client -> Driver -> Cluster Manager -> Workers -> Executors) is reflected directly in the UI. The Driver coordinates the job while Executors on Worker nodes perform the computations (although in this class all processes may reside on a single node).</p>
<p style="margin: 8px 0 0 0; color: #333">The <code>groupBy</code> forced a shuffle, so the job has two stages: one to compute the keys and one to aggregate after the exchange. This matches the DAG model from the lecture.</p>
</div>

In [0]:
%python
# Release the Cached DataFrame
# Free up executor memory by unpersisting cached objects
result.unpersist()

## Summary

- The `SparkSession` is created for you in Databricks as `spark`; the legacy `SparkContext` (`sc`) is exposed only on dedicated-access classic clusters
- Standard-access (shared) and Serverless compute use Spark Connect, where `sparkContext`, `sc`, and the RDD API are not available
- A shuffle operation splits a job into multiple stages, visible in the Spark UI DAG
- The Jobs, Executors, and Storage tabs of the Spark UI map directly to the runtime architecture components from the lecture

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>